# Block 3: Model 1 Road Closure Classifier

This notebook trains the first supervised model in the EventOps AI pipeline.

**Goal:** Predict whether a traffic event will require road closure at event creation time.

Input:

- `outputs/feature_engineering/model_ready_road_closure.csv`
- `outputs/feature_engineering/model_ready_duration_band.csv`

Outputs:

- `outputs/model1_v2/model1_v2_road_closure_xgb_model.pkl`
- `outputs/model1_v2/model1_v2_test_predictions.csv`
- `outputs/model1_v2/model1_v2_feature_importance.csv`
- `outputs/model1_v2/model1_v2_metrics.json`
- `outputs/model1_v2/model1_v2_duration_band_with_road_closure_features.csv`

The last file is important for Block 4: it adds `road_closure_probability` to duration-band features for stacked modeling.

## 1. Imports and Paths

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_selection import SelectFromModel
from sklearn.utils import resample

from xgboost import XGBClassifier

try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    SMOTE_AVAILABLE = False

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 100)

RANDOM_STATE = 42

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'outputs' / 'feature_engineering').exists()), cwd)
FEATURE_DIR = PROJECT_ROOT / 'outputs' / 'feature_engineering'
ROAD_PATH = FEATURE_DIR / 'model_ready_road_closure.csv'
DURATION_PATH = FEATURE_DIR / 'model_ready_duration_band.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'model1_v2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Road closure data:', ROAD_PATH)
print('Duration data:', DURATION_PATH)
print('Output dir:', OUTPUT_DIR)
print('SMOTE available:', SMOTE_AVAILABLE)

Project root: /Users/astron_designer/GridLock_Phase2
Road closure data: /Users/astron_designer/GridLock_Phase2/outputs/feature_engineering/model_ready_road_closure.csv
Duration data: /Users/astron_designer/GridLock_Phase2/outputs/feature_engineering/model_ready_duration_band.csv
Output dir: /Users/astron_designer/GridLock_Phase2/outputs/model1_v2


## 2. Load Model-Ready Data

In [2]:
road_df = pd.read_csv(ROAD_PATH, low_memory=False)
duration_df = pd.read_csv(DURATION_PATH, low_memory=False)

print('Road closure shape:', road_df.shape)
print('Duration-band shape:', duration_df.shape)
print('\nTarget distribution:')
display(road_df['target_road_closure'].value_counts().to_frame('count'))
print('\nTarget percentages:')
display((road_df['target_road_closure'].value_counts(normalize=True) * 100).round(2).to_frame('percent'))
display(road_df.head())

Road closure shape: (8173, 536)
Duration-band shape: (2764, 536)

Target distribution:


,count
target_road_closure,
0,7497
1,676



Target percentages:


,percent
target_road_closure,
0,91.73
1,8.27


,latitude,longitude,valid_start_coordinate,has_start_location,distance_to_city_center_km,start_hour,start_dayofweek,start_weekofyear,is_weekend,is_peak_hour,is_night,hour_sin,hour_cos,day_sin,day_cos,report_lag_minutes_clipped,report_lag_hours_clipped,report_lag_is_negative,reporting_delay_minutes,text_length,description_char_length,description_word_count,has_non_ascii_text,has_kannada_text,has_accident_word,has_breakdown_word,has_water_word,has_construction_word,has_event_word,has_blocked_word,has_jam_word,has_vip_word,has_location_hint_word,has_emergency_word,is_planned_event,is_public_or_vip_event,is_breakdown_event,is_accident_event,is_weather_or_visibility_event,is_road_condition_event,has_vehicle_type,is_truck,is_bus,is_two_wheeler,is_heavy_vehicle,has_cargo_material,age_of_truck,truck_age_missing,past_count_event_cause,past_closure_rate_event_cause,past_count_corridor,past_closure_rate_corridor,past_count_zone,past_closure_rate_zone,past_count_junction,past_closure_rate_junction,past_count_police_station,past_closure_rate_police_station,past_count_location_grid,past_closure_rate_location_grid,past_count_event_cause_corridor,past_closure_rate_event_cause_corridor,location_grid_12_905_77_602,location_grid_12_919_77_589,location_grid_12_925_77_618,location_grid_12_927_77_621,location_grid_12_928_77_621,location_grid_12_931_77_687,location_grid_12_939_77_747,location_grid_12_942_77_747,...,corridor_peak_interaction_bellary_road_1_morning_peak,corridor_peak_interaction_bellary_road_1_night,corridor_peak_interaction_bellary_road_1_off_peak,corridor_peak_interaction_bellary_road_2_morning_peak,corridor_peak_interaction_bellary_road_2_night,corridor_peak_interaction_bellary_road_2_off_peak,corridor_peak_interaction_cbd_2_night,corridor_peak_interaction_cbd_2_off_peak,corridor_peak_interaction_hennur_main_road_night,corridor_peak_interaction_hennur_main_road_off_peak,corridor_peak_interaction_hosur_road_morning_peak,corridor_peak_interaction_hosur_road_night,corridor_peak_interaction_hosur_road_off_peak,corridor_peak_interaction_irr_thanisandra_road_night,corridor_peak_interaction_irr_thanisandra_road_off_peak,corridor_peak_interaction_magadi_road_morning_peak,corridor_peak_interaction_magadi_road_night,corridor_peak_interaction_magadi_road_off_peak,corridor_peak_interaction_mysore_road_morning_peak,corridor_peak_interaction_mysore_road_night,corridor_peak_interaction_mysore_road_off_peak,corridor_peak_interaction_non_corridor_evening_peak,corridor_peak_interaction_non_corridor_morning_peak,corridor_peak_interaction_non_corridor_night,corridor_peak_interaction_non_corridor_off_peak,corridor_peak_interaction_non_corridor_unknown,corridor_peak_interaction_old_airport_road_night,corridor_peak_interaction_old_airport_road_off_peak,corridor_peak_interaction_old_madras_road_morning_peak,corridor_peak_interaction_old_madras_road_night,corridor_peak_interaction_old_madras_road_off_peak,corridor_peak_interaction_orr_east_1_morning_peak,corridor_peak_interaction_orr_east_1_night,corridor_peak_interaction_orr_east_1_off_peak,corridor_peak_interaction_orr_east_2_morning_peak,corridor_peak_interaction_orr_east_2_night,corridor_peak_interaction_orr_east_2_off_peak,corridor_peak_interaction_orr_north_1_morning_peak,corridor_peak_interaction_orr_north_1_night,corridor_peak_interaction_orr_north_1_off_peak,corridor_peak_interaction_orr_north_2_morning_peak,corridor_peak_interaction_orr_north_2_night,corridor_peak_interaction_orr_north_2_off_peak,corridor_peak_interaction_orr_west_1_morning_peak,corridor_peak_interaction_orr_west_1_night,corridor_peak_interaction_orr_west_1_off_peak,corridor_peak_interaction_other_rare,corridor_peak_interaction_tumkur_road_morning_peak,corridor_peak_interaction_tumkur_road_night,corridor_peak_interaction_tumkur_road_off_peak,corridor_peak_interaction_varthur_road_night,corridor_peak_interaction_varthur_road_off_peak,corridor_peak_interaction_west_of_chord_road_night,corridor_peak_interaction_west_of_chord_road_of

## 3. Feature Matrix and Target

The feature-engineering block already removed future/leakage columns. Here we only separate features from the road-closure target.

In [3]:
target_col = 'target_road_closure'
assert target_col in road_df.columns, f'Missing target column: {target_col}'

X = road_df.drop(columns=[target_col]).copy()
y = road_df[target_col].astype(int)

# Coerce all columns to numeric. Feature engineering should already make them numeric, but this keeps training robust.
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce')
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True)).fillna(0)

feature_cols = X.columns.tolist()

print('Feature matrix:', X.shape)
print('Positive rate:', round(float(y.mean()), 4))
print('Missing values after imputation:', int(X.isna().sum().sum()))

Feature matrix: (8173, 535)
Positive rate: 0.0827
Missing values after imputation: 0


## 4. Train / Validation / Test Split

We use a stratified 70/15/15 split to preserve the rare road-closure class in each split.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print('Train (before resampling):', X_train.shape, y_train.value_counts().to_dict())
print('Val:  ', X_val.shape, y_val.value_counts().to_dict())
print('Test: ', X_test.shape, y_test.value_counts().to_dict())

# ── Oversample minority class in training set ──
if SMOTE_AVAILABLE:
    smote = SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.4)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    print('\nTrain (after SMOTE):', X_train_resampled.shape, pd.Series(y_train_resampled).value_counts().to_dict())
else:
    # Fallback: random oversample minority to ~40% of majority
    majority = X_train[y_train == 0]
    minority = X_train[y_train == 1]
    target_minority_count = int(len(majority) * 0.4)
    minority_upsampled = resample(
        minority,
        replace=True,
        n_samples=target_minority_count,
        random_state=RANDOM_STATE,
    )
    X_train_resampled = pd.concat([majority, minority_upsampled])
    y_train_resampled = pd.concat([y_train[y_train == 0], pd.Series(1, index=minority_upsampled.index[:target_minority_count])])
    # Re-align index
    y_train_resampled = pd.Series(
        np.concatenate([np.zeros(len(majority)), np.ones(target_minority_count)]).astype(int)
    )
    X_train_resampled = X_train_resampled.reset_index(drop=True)
    y_train_resampled = y_train_resampled.reset_index(drop=True)
    print('\nTrain (after random oversample):', X_train_resampled.shape, y_train_resampled.value_counts().to_dict())

Train: (5721, 535) {0: 5248, 1: 473}
Val:   (1226, 535) {0: 1124, 1: 102}
Test:  (1226, 535) {0: 1125, 1: 101}


## 5. Baseline Model

A dummy classifier gives the minimum bar. A balanced Random Forest gives a stronger non-boosted baseline.

In [5]:
dummy = DummyClassifier(strategy='prior', random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
dummy_val_proba = dummy.predict_proba(X_val)[:, 1]

dummy_metrics = {
    'roc_auc': roc_auc_score(y_val, dummy_val_proba),
    'pr_auc': average_precision_score(y_val, dummy_val_proba),
}
print('Dummy baseline:', dummy_metrics)

rf_model = RandomForestClassifier(
    n_estimators=400,
    min_samples_leaf=2,
    class_weight='balanced_subsample',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
rf_val_proba = rf_model.predict_proba(X_val)[:, 1]
rf_val_pred = (rf_val_proba >= 0.5).astype(int)

print('\nRandom Forest Validation')
print('ROC-AUC:', round(roc_auc_score(y_val, rf_val_proba), 4))
print('PR-AUC:', round(average_precision_score(y_val, rf_val_proba), 4))
print('F1:', round(f1_score(y_val, rf_val_pred), 4))
print(classification_report(y_val, rf_val_pred, target_names=['No Closure', 'Road Closure']))

Dummy baseline: {'roc_auc': 0.5, 'pr_auc': 0.08319738988580751}



Random Forest Validation
ROC-AUC: 0.8387
PR-AUC: 0.4821
F1: 0.4267
              precision    recall  f1-score   support

  No Closure       0.94      0.99      0.96      1124
Road Closure       0.67      0.31      0.43       102

    accuracy                           0.93      1226
   macro avg       0.80      0.65      0.69      1226
weighted avg       0.92      0.93      0.92      1226



## 6. Main Model: XGBoost

Road closure is imbalanced, so we use `scale_pos_weight` to give the positive class more influence.

In [ ]:
neg_count = int((y_train_resampled == 0).sum())
pos_count = int((y_train_resampled == 1).sum())
scale_pos_weight = neg_count / max(pos_count, 1)
print('scale_pos_weight (after SMOTE):', round(scale_pos_weight, 3))

# ── IMPROVED: Feature selection using a quick pre-fit ──
pre_xgb = XGBClassifier(
    objective='binary:logistic', eval_metric='aucpr',
    n_estimators=200, max_depth=4, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist',
)
pre_xgb.fit(X_train_resampled, y_train_resampled, verbose=False)

selector = SelectFromModel(pre_xgb, threshold='0.5*mean', prefit=True)
selected_mask = selector.get_support()
selected_features = [f for f, s in zip(feature_cols, selected_mask) if s]
print(f'Feature selection: {len(selected_features)} / {len(feature_cols)} features retained')

X_train_sel = X_train_resampled[selected_features]
X_val_sel = X_val[selected_features]
X_test_sel = X_test[selected_features]

# ── IMPROVED: XGBoost with Stratified K-Fold CV for robust threshold ──
xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    n_estimators=800,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.80,
    colsample_bytree=0.80,
    min_child_weight=3,
    reg_lambda=5.0,
    reg_alpha=0.5,
    gamma=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method='hist',
)

xgb_model.fit(
    X_train_sel,
    y_train_resampled,
    eval_set=[(X_val_sel, y_val)],
    verbose=False,
)

val_proba = xgb_model.predict_proba(X_val_sel)[:, 1]
print('\nXGBoost Validation (selected features)')
print('ROC-AUC:', round(roc_auc_score(y_val, val_proba), 4))
print('PR-AUC:', round(average_precision_score(y_val, val_proba), 4))

# ── NEW: Cross-validated threshold selection ──
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_thresholds = []
cv_f1s = []

for fold_train_idx, fold_val_idx in skf.split(X_train_resampled[selected_features], y_train_resampled):
    fold_X_train = X_train_resampled[selected_features].iloc[fold_train_idx]
    fold_y_train = y_train_resampled.iloc[fold_train_idx] if hasattr(y_train_resampled, 'iloc') else y_train_resampled[fold_train_idx]
    fold_X_val = X_train_resampled[selected_features].iloc[fold_val_idx]
    fold_y_val = y_train_resampled.iloc[fold_val_idx] if hasattr(y_train_resampled, 'iloc') else y_train_resampled[fold_val_idx]
    
    fold_model = XGBClassifier(
        objective='binary:logistic', eval_metric='aucpr',
        n_estimators=500, max_depth=5, learning_rate=0.03,
        subsample=0.80, colsample_bytree=0.80, min_child_weight=3,
        reg_lambda=5.0, reg_alpha=0.5, gamma=0.1,
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist',
    )
    fold_model.fit(fold_X_train, fold_y_train, verbose=False)
    fold_proba = fold_model.predict_proba(fold_X_val)[:, 1]
    
    prec, rec, threshs = precision_recall_curve(fold_y_val, fold_proba)
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    best_idx = int(np.argmax(f1s))
    cv_thresholds.append(float(threshs[best_idx]))
    cv_f1s.append(float(f1s[best_idx]))

cv_optimal_threshold = float(np.median(cv_thresholds))
print(f'\nCV thresholds: {[round(t, 4) for t in cv_thresholds]}')
print(f'CV F1 scores:  {[round(f, 4) for f in cv_f1s]}')
print(f'Median CV threshold: {round(cv_optimal_threshold, 4)}')

scale_pos_weight: 11.095


XGBoost Validation
ROC-AUC: 0.8449
PR-AUC: 0.4807


## 7. Threshold Tuning (uses CV-derived + validation PR curve)

# Also tune on the holdout validation set for a second opinion
prec_v, rec_v, threshs_v = precision_recall_curve(y_val, val_proba)
f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-12)
val_optimal_idx = int(np.argmax(f1_v))
val_optimal_threshold = float(threshs_v[val_optimal_idx])

# Use average of CV median and val-optimal for robustness
optimal_threshold = (cv_optimal_threshold + val_optimal_threshold) / 2.0

print(f'Val-only optimal threshold: {round(val_optimal_threshold, 4)} (F1={round(float(f1_v[val_optimal_idx]), 4)})')
print(f'CV median threshold:        {round(cv_optimal_threshold, 4)}')
print(f'Final blended threshold:    {round(optimal_threshold, 4)}')

In [7]:
precision, recall, thresholds = precision_recall_curve(y_val, val_proba)
f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
best_idx = int(np.argmax(f1_scores))
optimal_threshold = float(thresholds[best_idx])

val_pred = (val_proba >= optimal_threshold).astype(int)

print('Optimal threshold:', round(optimal_threshold, 4))
print('Validation precision:', round(precision_score(y_val, val_pred, zero_division=0), 4))
print('Validation recall:', round(recall_score(y_val, val_pred, zero_division=0), 4))
print('Validation F1:', round(f1_score(y_val, val_pred), 4))
print('Validation balanced accuracy:', round(balanced_accuracy_score(y_val, val_pred), 4))
print('\nClassification Report:')
print(classification_report(y_val, val_pred, target_names=['No Closure', 'Road Closure']))
print('\nConfusion Matrix:')
print(confusion_matrix(y_val, val_pred))

Optimal threshold: 0.7108
Validation precision: 0.5789
Validation recall: 0.4314
Validation F1: 0.4944
Validation balanced accuracy: 0.7015

Classification Report:
              precision    recall  f1-score   support

  No Closure       0.95      0.97      0.96      1124
Road Closure       0.58      0.43      0.49       102

    accuracy                           0.93      1226
   macro avg       0.76      0.70      0.73      1226
weighted avg       0.92      0.93      0.92      1226


Confusion Matrix:
[[1092   32]
 [  58   44]]


test_proba = xgb_model.predict_proba(X_test_sel)[:, 1]
test_pred = (test_proba >= optimal_threshold).astype(int)

test_roc_auc = roc_auc_score(y_test, test_proba)
test_pr_auc = average_precision_score(y_test, test_proba)
test_f1 = f1_score(y_test, test_pred)
test_precision = precision_score(y_test, test_pred)
test_recall = recall_score(y_test, test_pred)
test_accuracy = accuracy_score(y_test, test_pred)
test_balanced_acc = balanced_accuracy_score(y_test, test_pred)

metrics = {
    'model': 'XGBoost',
    'threshold': float(optimal_threshold),
    'cv_threshold_median': float(cv_optimal_threshold),
    'validation_roc_auc': float(roc_auc_score(y_val, val_proba)),
    'validation_pr_auc': float(average_precision_score(y_val, val_proba)),
    'validation_f1': float(f1_score(y_val, (val_proba >= optimal_threshold).astype(int))),
    'validation_recall': float(recall_score(y_val, (val_proba >= optimal_threshold).astype(int))),
    'test_roc_auc': float(test_roc_auc),
    'test_pr_auc': float(test_pr_auc),
    'test_accuracy': float(test_accuracy),
    'test_balanced_accuracy': float(test_balanced_acc),
    'test_precision': float(test_precision),
    'test_recall': float(test_recall),
    'test_f1': float(test_f1),
    'positive_rate': float(y.mean()),
    'feature_count': len(selected_features),
    'feature_count_before_selection': len(feature_cols),
    'smote_used': SMOTE_AVAILABLE,
}

print('FINAL TEST RESULTS (Model 1 v2 Improved)')
print('=' * 50)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

print('\nClassification Report:')
print(classification_report(y_test, test_pred, target_names=['No Closure', 'Road Closure']))
print('Confusion Matrix:')
print(confusion_matrix(y_test, test_pred))

In [8]:
test_proba = xgb_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= optimal_threshold).astype(int)

metrics = {
    'model': 'XGBoost',
    'threshold': optimal_threshold,
    'validation_roc_auc': float(roc_auc_score(y_val, val_proba)),
    'validation_pr_auc': float(average_precision_score(y_val, val_proba)),
    'validation_f1': float(f1_score(y_val, val_pred)),
    'validation_recall': float(recall_score(y_val, val_pred, zero_division=0)),
    'test_roc_auc': float(roc_auc_score(y_test, test_proba)),
    'test_pr_auc': float(average_precision_score(y_test, test_proba)),
    'test_accuracy': float(accuracy_score(y_test, test_pred)),
    'test_balanced_accuracy': float(balanced_accuracy_score(y_test, test_pred)),
    'test_precision': float(precision_score(y_test, test_pred, zero_division=0)),
    'test_recall': float(recall_score(y_test, test_pred, zero_division=0)),
    'test_f1': float(f1_score(y_test, test_pred)),
    'positive_rate': float(y.mean()),
    'feature_count': int(X.shape[1]),
}

print('XGBoost Test Evaluation')
for key, value in metrics.items():
    if isinstance(value, float):
        print(f'{key}: {value:.4f}')
    else:
        print(f'{key}: {value}')

print('\nClassification Report:')
print(classification_report(y_test, test_pred, target_names=['No Closure', 'Road Closure']))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, test_pred))

XGBoost Test Evaluation
model: XGBoost
threshold: 0.7108
validation_roc_auc: 0.8449
validation_pr_auc: 0.4807
validation_f1: 0.4944
validation_recall: 0.4314
test_roc_auc: 0.8503
test_pr_auc: 0.5333
test_accuracy: 0.9307
test_balanced_accuracy: 0.7144
test_precision: 0.6053
test_recall: 0.4554
test_f1: 0.5198
positive_rate: 0.0827
feature_count: 535

Classification Report:
              precision    recall  f1-score   support

  No Closure       0.95      0.97      0.96      1125
Road Closure       0.61      0.46      0.52       101

    accuracy                           0.93      1226
   macro avg       0.78      0.71      0.74      1226
weighted avg       0.92      0.93      0.93      1226


Confusion Matrix:
[[1095   30]
 [  55   46]]


feature_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': xgb_model.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)

print(f'Top 40 features (of {len(selected_features)} selected):')
display(feature_importance.head(40))

In [9]:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)

display(feature_importance.head(40))

,feature,importance
0,past_closure_rate_event_cause,0.019245
1,location_grid_other_rare,0.014824
2,is_breakdown_event,0.013582
3,cause_heavy_vehicle_interaction_construction_h...,0.011363
4,has_jam_word,0.010717
5,corridor_grouped_non_corridor,0.009799
6,event_cause_corridor_construction_orr_east_2,0.009524
7,peak_period_unknown,0.009099
8,corridor_grouped_orr_east_2,0.008700
9,event_cause_grouped_water_logging,0.008642


test_predictions = pd.DataFrame({
    'source_row_index': X_test.index,
    'actual_label': y_test.values,
    'predicted_label': test_pred,
    'road_closure_probability': test_proba,
    'road_closure_percentage': test_proba * 100,
}).sort_values('source_row_index').reset_index(drop=True)

# ── NEW: Calibrate the model for better probability estimates in Model 2 ──
calibrated_model = CalibratedClassifierCV(xgb_model, method='isotonic', cv=3)
calibrated_model.fit(X_val_sel, y_val)

model_artifact = {
    'model': xgb_model,
    'calibrated_model': calibrated_model,
    'threshold': optimal_threshold,
    'feature_cols': selected_features,
    'all_feature_cols': feature_cols,
    'selected_features': selected_features,
    'metrics': metrics,
}

model_path = OUTPUT_DIR / 'model1_v2_road_closure_xgb_model.pkl'
predictions_path = OUTPUT_DIR / 'model1_v2_test_predictions.csv'
importance_path = OUTPUT_DIR / 'model1_v2_feature_importance.csv'
metrics_path = OUTPUT_DIR / 'model1_v2_metrics.json'

joblib.dump(model_artifact, model_path)
test_predictions.to_csv(predictions_path, index=False)
feature_importance.to_csv(importance_path, index=False)
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved model:', model_path)
print('Saved predictions:', predictions_path)
print('Saved feature importance:', importance_path)
print('Saved metrics:', metrics_path)
display(test_predictions.head())

In [10]:
test_predictions = pd.DataFrame({
    'source_row_index': X_test.index,
    'actual_label': y_test.values,
    'predicted_label': test_pred,
    'road_closure_probability': test_proba,
    'road_closure_percentage': test_proba * 100,
}).sort_values('source_row_index').reset_index(drop=True)

model_artifact = {
    'model': xgb_model,
    'threshold': optimal_threshold,
    'feature_cols': feature_cols,
    'metrics': metrics,
}

model_path = OUTPUT_DIR / 'model1_v2_road_closure_xgb_model.pkl'
predictions_path = OUTPUT_DIR / 'model1_v2_test_predictions.csv'
importance_path = OUTPUT_DIR / 'model1_v2_feature_importance.csv'
metrics_path = OUTPUT_DIR / 'model1_v2_metrics.json'

joblib.dump(model_artifact, model_path)
test_predictions.to_csv(predictions_path, index=False)
feature_importance.to_csv(importance_path, index=False)
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved model:', model_path)
print('Saved predictions:', predictions_path)
print('Saved feature importance:', importance_path)
print('Saved metrics:', metrics_path)
display(test_predictions.head())

Saved model: /Users/astron_designer/GridLock_Phase2/outputs/model1_v2/model1_v2_road_closure_xgb_model.pkl
Saved predictions: /Users/astron_designer/GridLock_Phase2/outputs/model1_v2/model1_v2_test_predictions.csv
Saved feature importance: /Users/astron_designer/GridLock_Phase2/outputs/model1_v2/model1_v2_feature_importance.csv
Saved metrics: /Users/astron_designer/GridLock_Phase2/outputs/model1_v2/model1_v2_metrics.json


,source_row_index,actual_label,predicted_label,road_closure_probability,road_closure_percentage
0,1,0,0,0.479556,47.955627
1,3,1,1,0.902975,90.297508
2,4,0,0,0.692699,69.269943
3,5,0,0,0.044654,4.465383
4,14,0,0,0.050042,5.004175


duration_features = duration_df.drop(columns=['duration_band']).copy()

# Align duration features to the exact Model 1 feature columns (all features before selection).
missing_cols = [c for c in feature_cols if c not in duration_features.columns]
extra_cols = [c for c in duration_features.columns if c not in feature_cols]

for col in missing_cols:
    duration_features[col] = 0

duration_features = duration_features[feature_cols].copy()
for col in duration_features.columns:
    duration_features[col] = pd.to_numeric(duration_features[col], errors='coerce')
duration_features = duration_features.replace([np.inf, -np.inf], np.nan)
duration_features = duration_features.fillna(X_train.median(numeric_only=True)).fillna(0)

# Use selected features for prediction
duration_features_sel = duration_features[selected_features].copy()

# ── NEW: Use calibrated model for better probability estimates ──
duration_closure_proba = calibrated_model.predict_proba(duration_features_sel)[:, 1]
duration_closure_label = (duration_closure_proba >= optimal_threshold).astype(int)

model2_stacked_df = duration_df.copy()
model2_stacked_df['road_closure_probability'] = duration_closure_proba
model2_stacked_df['road_closure_predicted_label'] = duration_closure_label

# Keep probability for modeling; predicted label is useful for audit but less ideal as a feature.
model2_clean_stacked_df = model2_stacked_df.drop(columns=['road_closure_predicted_label'])

stacked_full_path = OUTPUT_DIR / 'model1_v2_duration_band_with_road_closure_audit.csv'
stacked_clean_path = OUTPUT_DIR / 'model1_v2_duration_band_with_road_closure_features.csv'

model2_stacked_df.to_csv(stacked_full_path, index=False)
model2_clean_stacked_df.to_csv(stacked_clean_path, index=False)

print('Missing duration feature cols filled:', len(missing_cols))
print('Extra duration feature cols ignored:', len(extra_cols))
print('Saved audit stacked file:', stacked_full_path)
print('Saved clean stacked feature file:', stacked_clean_path)
print('Stacked clean shape:', model2_clean_stacked_df.shape)
print('Has road_closure_probability:', 'road_closure_probability' in model2_clean_stacked_df.columns)
print('Using calibrated probabilities: True')
display(model2_clean_stacked_df[['road_closure_probability', 'duration_band']].head())

In [11]:
duration_features = duration_df.drop(columns=['duration_band']).copy()

# Align duration features to the exact Model 1 feature columns.
missing_cols = [c for c in feature_cols if c not in duration_features.columns]
extra_cols = [c for c in duration_features.columns if c not in feature_cols]

for col in missing_cols:
    duration_features[col] = 0

duration_features = duration_features[feature_cols].copy()
for col in duration_features.columns:
    duration_features[col] = pd.to_numeric(duration_features[col], errors='coerce')
duration_features = duration_features.replace([np.inf, -np.inf], np.nan)
duration_features = duration_features.fillna(X_train.median(numeric_only=True)).fillna(0)

duration_closure_proba = xgb_model.predict_proba(duration_features)[:, 1]
duration_closure_label = (duration_closure_proba >= optimal_threshold).astype(int)

model2_stacked_df = duration_df.copy()
model2_stacked_df['road_closure_probability'] = duration_closure_proba
model2_stacked_df['road_closure_predicted_label'] = duration_closure_label

# Keep probability for modeling; predicted label is useful for audit but less ideal as a feature.
model2_clean_stacked_df = model2_stacked_df.drop(columns=['road_closure_predicted_label'])

stacked_full_path = OUTPUT_DIR / 'model1_v2_duration_band_with_road_closure_audit.csv'
stacked_clean_path = OUTPUT_DIR / 'model1_v2_duration_band_with_road_closure_features.csv'

model2_stacked_df.to_csv(stacked_full_path, index=False)
model2_clean_stacked_df.to_csv(stacked_clean_path, index=False)

print('Missing duration feature cols filled:', len(missing_cols))
print('Extra duration feature cols ignored:', len(extra_cols))
print('Saved audit stacked file:', stacked_full_path)
print('Saved clean stacked feature file:', stacked_clean_path)
print('Stacked clean shape:', model2_clean_stacked_df.shape)
print('Has road_closure_probability:', 'road_closure_probability' in model2_clean_stacked_df.columns)
display(model2_clean_stacked_df[['road_closure_probability', 'duration_band']].head())

Missing duration feature cols filled: 0
Extra duration feature cols ignored: 0
Saved audit stacked file: /Users/astron_designer/GridLock_Phase2/outputs/model1_v2/model1_v2_duration_band_with_road_closure_audit.csv
Saved clean stacked feature file: /Users/astron_designer/GridLock_Phase2/outputs/model1_v2/model1_v2_duration_band_with_road_closure_features.csv
Stacked clean shape: (2764, 537)
Has road_closure_probability: True


,road_closure_probability,duration_band
0,0.479556,short
1,0.692699,short
2,0.925349,long
3,0.248171,short
4,0.954939,long


print('=' * 72)
print('BLOCK 3 COMPLETE: MODEL 1 ROAD CLOSURE CLASSIFIER (IMPROVED)')
print('=' * 72)
print('Model: XGBoost + SMOTE + Feature Selection + Calibration')
print('Features (selected):', len(selected_features))
print('Features (original):', len(feature_cols))
print('Threshold:', round(optimal_threshold, 4))
print('Test PR-AUC:', round(metrics['test_pr_auc'], 4))
print('Test ROC-AUC:', round(metrics['test_roc_auc'], 4))
print('Test F1:', round(metrics['test_f1'], 4))
print('Test Recall:', round(metrics['test_recall'], 4))
print('Test Precision:', round(metrics['test_precision'], 4))
print('SMOTE used:', SMOTE_AVAILABLE)
print('Calibration: isotonic')
print('Saved stacked Block 4 input:', stacked_clean_path)
print('=' * 72)

In [12]:
print('=' * 72)
print('BLOCK 3 COMPLETE: MODEL 1 ROAD CLOSURE CLASSIFIER')
print('=' * 72)
print('Model: XGBoost')
print('Features:', X.shape[1])
print('Threshold:', round(optimal_threshold, 4))
print('Test PR-AUC:', round(metrics['test_pr_auc'], 4))
print('Test ROC-AUC:', round(metrics['test_roc_auc'], 4))
print('Test F1:', round(metrics['test_f1'], 4))
print('Test Recall:', round(metrics['test_recall'], 4))
print('Saved stacked Block 4 input:', stacked_clean_path)
print('=' * 72)

BLOCK 3 COMPLETE: MODEL 1 ROAD CLOSURE CLASSIFIER
Model: XGBoost
Features: 535
Threshold: 0.7108
Test PR-AUC: 0.5333
Test ROC-AUC: 0.8503
Test F1: 0.5198
Test Recall: 0.4554
Saved stacked Block 4 input: /Users/astron_designer/GridLock_Phase2/outputs/model1_v2/model1_v2_duration_band_with_road_closure_features.csv
